# Fleet Dimensioning: Monte Carlo Simulation

This notebook answers the central business question: **how many vehicles does the carrier need to absorb the e-commerce client's pickup operation?**

We use Monte Carlo simulation rather than deterministic calculation because two key inputs are uncertain in real logistics:

| Uncertainty | Modeled as |
|---|---|
| Daily pickup volume | ±10% uniform variation |
| Time per pickup | ±10% uniform variation |
| Traffic factor | 1.0×–1.3× (São Paulo congestion range) |

With 5,000 simulation runs per (cluster × vehicle-capacity × trips/day) combination, we get a P5/P50/P95 distribution of required vehicles — giving the carrier a risk-aware recommendation rather than a single brittle number.

---

## 1. Setup

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

sys.path.append('../src')
from logistics_optimizer import LogisticsOptimizer

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

## 2. Load Data and Run Clustering

This notebook is self-contained — it re-runs the clustering from notebook 02 so it can be executed independently.

In [ ]:
try:
    df = pd.read_excel('../data/processed/Dados_logistica.xlsx')
except FileNotFoundError:
    df = pd.read_csv('../data/processed/enderecos_com_coordenadas.csv')

volume_col = next((c for c in df.columns if 'VOLUME' in c.upper() and 'PICKUP' in c.upper()), None)
if volume_col and volume_col != 'VOLUME_PICKUP':
    df = df.rename(columns={volume_col: 'VOLUME_PICKUP'})

optimizer = LogisticsOptimizer()
optimizer.load_data(df=df)
clusters = optimizer.perform_clustering(n_clusters=3)
print(f"\n{len(optimizer.df)} pickup points across {clusters.shape[0]} clusters")

## 3. Monte Carlo Simulation

**Parameters:**
- `n_simulations = 5,000` per scenario
- Vehicle capacities tested: 100, 150, 300 packages/vehicle
- Trips per day: 1 or 2
- Working day: 8 hours × 80% efficiency cap
- Service time: 15 min/pickup point + 3 min/km travel (before traffic factor)

Each simulation draw independently samples volume variation, service time variation, and a traffic multiplier, then computes how many vehicles are needed to clear the cluster's demand within a working day.

In [ ]:
simulation = optimizer.monte_carlo_simulation(n_simulations=5000)
print(f"\nSimulation scenarios computed: {len(simulation)}")
simulation.head(12)

## 4. Fleet Recommendations by Scenario

Three operational scenarios are compared:

| Scenario | Vehicle Capacity | Trips/Day | Rationale |
|---|---|---|---|
| **Conservative** | 100 packages | 1 | Small vans, minimal capital commitment |
| **Moderate** ✓ | 150 packages | 1 | Mid-size cargo vans, recommended balance |
| **Optimistic** | 300 packages | 2 | Large trucks doing 2 trips — needs operational maturity |

In [ ]:
recommendations = optimizer.generate_fleet_recommendations()
recommendations

## 5. Scenario Comparison Chart

In [ ]:
scenario_map = {
    (100, 1): 'Conservative',
    (150, 1): 'Moderate',
    (300, 2): 'Optimistic'
}

scenario_totals = {}
for (cap, trips), label in scenario_map.items():
    subset = simulation[(simulation['capacity'] == cap) & (simulation['trips'] == trips)]
    total_p5 = subset['min_vehicles'].sum()
    total_p50 = subset['median_vehicles'].sum()
    total_p95 = subset['max_vehicles'].sum()
    scenario_totals[label] = (total_p5, total_p50, total_p95)

labels = list(scenario_totals.keys())
p5s = [v[0] for v in scenario_totals.values()]
p50s = [v[1] for v in scenario_totals.values()]
p95s = [v[2] for v in scenario_totals.values()]

x = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
bars_p5  = ax.bar(x - width, p5s,  width, label='P5 (optimistic)',  color='#2ecc71', alpha=0.85)
bars_p50 = ax.bar(x,         p50s, width, label='P50 (median)',     color='#3498db', alpha=0.85)
bars_p95 = ax.bar(x + width, p95s, width, label='P95 (conservative)', color='#e74c3c', alpha=0.85)

for bar in [bars_p5, bars_p50, bars_p95]:
    for b in bar:
        ax.text(b.get_x() + b.get_width()/2., b.get_height() + 0.3,
                str(int(b.get_height())), ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('Total Vehicles (all clusters)')
ax.set_title('Fleet Size by Operational Scenario — Monte Carlo P5/P50/P95')
ax.legend()
plt.tight_layout()
plt.show()

print("\nTotal fleet summary:")
for label, (p5, p50, p95) in scenario_totals.items():
    print(f"  {label:15s}: P5={p5}, P50={p50}, P95={p95}  (range: {p5}–{p95})")

## 6. Sensitivity Analysis: Excluding the Brás Zone

The Brás neighborhood in central São Paulo is a wholesale district with a very high density of high-volume sellers — 45 of the 270 pickup points are in Brás alone, and they account for a disproportionate share of total package volume.

**Business question:** Could the carrier onboard the e-commerce client *excluding Brás* as a first phase, and add Brás later once the operation is running? This scenario analysis quantifies the fleet reduction that phased onboarding would enable.

In [ ]:
df_no_bras = df[df['AREA'] != 'Brás'].copy()
volume_col_nb = next((c for c in df_no_bras.columns if 'VOLUME' in c.upper() and 'PICKUP' in c.upper()), None)
if volume_col_nb and volume_col_nb != 'VOLUME_PICKUP':
    df_no_bras = df_no_bras.rename(columns={volume_col_nb: 'VOLUME_PICKUP'})

print(f"Full dataset: {len(df)} points")
print(f"Excluding Brás: {len(df_no_bras)} points ({len(df) - len(df_no_bras)} removed)")

if 'VOLUME_PICKUP' in df.columns and 'VOLUME_PICKUP' in df_no_bras.columns:
    print(f"Volume removed: {df['VOLUME_PICKUP'].sum() - df_no_bras['VOLUME_PICKUP'].sum():,} packages ")
    print(f"  ({(df['VOLUME_PICKUP'].sum() - df_no_bras['VOLUME_PICKUP'].sum()) / df['VOLUME_PICKUP'].sum() * 100:.1f}% of total)")

In [ ]:
optimizer_no_bras = LogisticsOptimizer()
optimizer_no_bras.load_data(df=df_no_bras)
optimizer_no_bras.perform_clustering(n_clusters=3)
sim_no_bras = optimizer_no_bras.monte_carlo_simulation(n_simulations=5000)
recs_no_bras = optimizer_no_bras.generate_fleet_recommendations()

In [ ]:
# Compare moderate scenario totals: full vs. no-Brás
def total_moderate(sim_df):
    s = sim_df[(sim_df['capacity'] == 150) & (sim_df['trips'] == 1)]
    return s['min_vehicles'].sum(), s['median_vehicles'].sum(), s['max_vehicles'].sum()

full_p5, full_p50, full_p95 = total_moderate(simulation)
nb_p5, nb_p50, nb_p95 = total_moderate(sim_no_bras)

print("=== Moderate Scenario (150 cap / 1 trip) — Full vs. Excluding Brás ===")
print(f"{'':25s} {'P5':>6} {'P50':>6} {'P95':>6}")
print(f"{'Full scope':25s} {full_p5:>6} {full_p50:>6} {full_p95:>6}")
print(f"{'Excluding Brás':25s} {nb_p5:>6} {nb_p50:>6} {nb_p95:>6}")
print(f"{'Difference':25s} {full_p5-nb_p5:>6} {full_p50-nb_p50:>6} {full_p95-nb_p95:>6}")

## 7. Export Results

In [ ]:
import os
os.makedirs('../outputs', exist_ok=True)

optimizer.export_fleet_recommendations('../outputs/fleet_recommendations.xlsx', add_timestamp=False)
optimizer.export_interactive_map('../outputs/cluster_map.html', add_timestamp=False)
optimizer.export_all_results('../outputs/full_analysis', add_timestamp=False)

print("\nAll outputs saved to ../outputs/")

## 8. Executive Summary

### Full Scope (270 pickup points, ~5,083 packages/day)

| Scenario | Vehicle Capacity | Trips/Day | Fleet Range |
|---|---|---|---|
| Conservative | 100 packages | 1 | see P5–P95 above |
| **Moderate (recommended)** | **150 packages** | **1** | **see P5–P95 above** |
| Optimistic | 300 packages | 2 | see P5–P95 above |

**Recommended scenario:** Moderate (150 packages/vehicle, 1 trip/day). This balances capital investment against operational complexity — 150-package cargo vans are widely available to rent/lease in the São Paulo metro market, and a single daily trip per vehicle keeps logistics coordination manageable for a carrier expanding into a new client.

### Phased Onboarding Option (Excluding Brás)

Starting without the Brás zone significantly reduces fleet requirements. Brás can be onboarded in phase 2 once the carrier has validated its operations with the e-commerce client.

### Key Risk Factor

The volume distribution is heavily right-skewed: most sellers dispatch 1–10 packages, but at least one seller dispatches up to 796 packages in a single pickup. This outlier drives a significant portion of fleet sizing for the cluster it belongs to. Negotiating a dedicated vehicle for ultra-high-volume sellers (>200 packages) outside the standard fleet would reduce uncertainty.